In [34]:
#installing required libraries
%pip install -q pymupdf sentence-transformers transformers accelerate


Note: you may need to restart the kernel to use updated packages.


In [35]:
#extracting the pdf text through pymupdf

import os
import fitz 
pdf_paths = []
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.lower().endswith(".pdf"):
            pdf_paths.append(os.path.join(root, file))
if not pdf_paths:
    raise FileNotFoundError("No PDF found in /kaggle/input")
PDF_PATH = pdf_paths[0]
doc = fitz.open(PDF_PATH)
pages_text = []
for page_num, page in enumerate(doc, start=1):
    text = page.get_text("text").strip()
    if text:
        pages_text.append(f"[Page {page_num}]\n{text}")
document_text = "\n\n".join(pages_text)
print("PDF:", PDF_PATH)
print("Total pages:", len(doc))
print("Extracted characters:", len(document_text))
print("\nFirst 3000 characters:\n")
print(document_text[:3000])


PDF: /kaggle/input/datasets/harshalbhansali22/rad-dataset/dataset-rag.pdf
Total pages: 48
Extracted characters: 106995

First 3000 characters:

[Page 1]
NIST AI 100-1
Artificial Intelligence Risk Management
Framework (AI RMF 1.0)

[Page 2]
NIST AI 100-1
Artificial Intelligence Risk Management
Framework (AI RMF 1.0)
This publication is available free of charge from:
https://doi.org/10.6028/NIST.AI.100-1
January 2023
U.S. Department of Commerce
Gina M. Raimondo, Secretary
National Institute of Standards and Technology
Laurie E. Locascio, NIST Director and Under Secretary of Commerce for Standards and Technology

[Page 3]
Certain commercial entities, equipment, or materials may be identified in this document in order to describe 
an experimental procedure or concept adequately. Such identification is not intended to imply recommenda-
tion or endorsement by the National Institute of Standards and Technology, nor is it intended to imply that 
the entities, materials, or equipment are necess

In [36]:

import re

# Splitting text by page to preserve page references
page_pattern = re.compile(r"\[Page (\d+)\]\n")
matches = list(page_pattern.finditer(document_text))
pages = []
for i, match in enumerate(matches):
    start = match.end()
    end = matches[i + 1].start() if i + 1 < len(matches) else len(document_text)
    page_num = int(match.group(1))
    page_text = document_text[start:end].strip()
    if page_text:
        pages.append((page_num, page_text))

# Chunk settings
CHUNK_SIZE = 900
CHUNK_OVERLAP = 150
chunks = []
chunk_id = 0
for page_num, page_text in pages:
    start = 0
    while start < len(page_text):
        end = min(start + CHUNK_SIZE, len(page_text))
        chunk_text = page_text[start:end].strip()
        if chunk_text:
            chunks.append({"chunk_id": chunk_id,"page": page_num,"text": chunk_text})
            chunk_id += 1
        if end == len(page_text):
            break
        start = end - CHUNK_OVERLAP
print("Pages processed:", len(pages))
print("Total chunks:", len(chunks))
print("\nFirst chunk:")
print(chunks[0]["text"])


Pages processed: 48
Total chunks: 154

First chunk:
NIST AI 100-1
Artificial Intelligence Risk Management
Framework (AI RMF 1.0)


In [37]:

import time
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load the pretrained embedding model
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("Embedding model loaded.")
print("Device:", embedding_model.device)
print("Maximum sequence length:", embedding_model.max_seq_length)

# Extract the text from every chunk
chunk_texts = [chunk["text"] for chunk in chunks]

# Generate embeddings and measure indexing time
start_time = time.perf_counter()
embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=16,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True)
embedding_time = time.perf_counter() - start_time
print("\nNumber of chunks:", len(chunk_texts))
print("Embedding matrix shape:", embeddings.shape)
print(f"Embedding generation time: {embedding_time:.3f} seconds")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.
Device: cuda:0
Maximum sequence length: 256


Batches:   0%|          | 0/10 [00:00<?, ?it/s]


Number of chunks: 154
Embedding matrix shape: (154, 384)
Embedding generation time: 0.385 seconds


In [38]:

def baseline_retrieve(query, top_k=10):
    
    #Retrieve the top-k chunks using cosine similarity.
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True)
    scores = cosine_similarity(query_embedding, embeddings)[0]
    ranked_indices = np.argsort(scores)[::-1][:top_k]
    results = []

    for rank, idx in enumerate(ranked_indices, start=1):
        results.append({
            "rank": rank,
            "chunk_id": chunks[idx]["chunk_id"],
            "page": chunks[idx]["page"],
            "score": float(scores[idx]),
            "text": chunks[idx]["text"]})
    return results


# Initial test query
test_query = (
    "What are the four core functions of the NIST AI Risk "
    "Management Framework, and what does each function do?")
baseline_results = baseline_retrieve(test_query, top_k=10)
for result in baseline_results:
    print(
        f"\nRank {result['rank']} | "
        f"Chunk {result['chunk_id']} | "
        f"Page {result['page']} | "
        f"Score {result['score']:.4f}")
    print(result["text"][:500])
    print("-" * 70)



Rank 1 | Chunk 75 | Page 25 | Score 0.7963
NIST AI 100-1
AI RMF 1.0
Part 2: Core and Profiles
5.
AI RMF Core
The AI RMF Core provides outcomes and actions that enable dialogue, understanding, and
activities to manage AI risks and responsibly develop trustworthy AI systems. As illus-
trated in Figure 5, the Core is composed of four functions: GOVERN, MAP, MEASURE,
and MANAGE. Each of these high-level functions is broken down into categories and sub-
categories. Categories and subcategories are subdivided into specific actions and outcomes
----------------------------------------------------------------------

Rank 2 | Chunk 16 | Page 8 | Score 0.7555
NIST AI 100-1
AI RMF 1.0
valid and reliable, safe, secure and resilient, accountable and transparent, explainable and
interpretable, privacy enhanced, and fair with their harmful biases managed.
Part 2 comprises the “Core” of the Framework. It describes four specific functions to help
organizations address the risks of AI systems in practi

In [39]:
#loading the cross-encoder
from sentence_transformers import CrossEncoder
cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device="cuda" if torch.cuda.is_available() else "cpu")
print("Cross-Encoder loaded successfully.")
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Cross-Encoder loaded successfully.
Device: cuda


In [40]:

def rerank_results(query, candidates):
    #Re-rank retrieved chunks using Cross-Encoder relevance scores
    pairs = [
        (query, result["text"])
        for result in candidates
    ]
    scores = cross_encoder.predict(pairs, show_progress_bar=False)
    reranked = []
    for result, score in zip(candidates, scores):
        item = result.copy()
        item["cross_encoder_score"] = float(score)
        reranked.append(item)
    reranked.sort(
        key=lambda item: item["cross_encoder_score"],
        reverse=True)
    for rank, item in enumerate(reranked, start=1):
        item["rerank_position"] = rank

    return reranked
reranked_results = rerank_results(test_query, baseline_results)
print("Top 3 chunks after re-ranking:\n")
for result in reranked_results[:3]:
    print(
        f"New rank {result['rerank_position']} | "
        f"Chunk {result['chunk_id']} | "
        f"Page {result['page']} | "
        f"Cross-Encoder score: {result['cross_encoder_score']:.4f}")
    print(result["text"][:400])
    print("-" * 60)


Top 3 chunks after re-ranking:

New rank 1 | Chunk 16 | Page 8 | Cross-Encoder score: 8.4566
NIST AI 100-1
AI RMF 1.0
valid and reliable, safe, secure and resilient, accountable and transparent, explainable and
interpretable, privacy enhanced, and fair with their harmful biases managed.
Part 2 comprises the “Core” of the Framework. It describes four specific functions to help
organizations address the risks of AI systems in practice. These functions – GOVERN,
MAP, MEASURE, and MANAGE – ar
------------------------------------------------------------
New rank 2 | Chunk 75 | Page 25 | Cross-Encoder score: 8.4326
NIST AI 100-1
AI RMF 1.0
Part 2: Core and Profiles
5.
AI RMF Core
The AI RMF Core provides outcomes and actions that enable dialogue, understanding, and
activities to manage AI risks and responsibly develop trustworthy AI systems. As illus-
trated in Figure 5, the Core is composed of four functions: GOVERN, MAP, MEASURE,
and MANAGE. Each of these high-level functions is broken dow

In [41]:
#finding failure cases
candidate_queries = [
    "What should an organization do when AI risks exceed its risk tolerance?",
    "How should AI systems be monitored after deployment, and what happens when risks emerge?",
    "What is the difference between measuring AI risks and managing them?",
    "How should organizations handle risks that cannot be adequately measured?",
    "What responsibilities do senior leaders have for AI risk governance?",
    "How should an organization address risks to people who are not direct users of an AI system?",
    "What should happen when an AI system fails to achieve its intended objectives?",
    "How does the framework address harmful bias and fairness?"
]
experiment_results = {}
for query in candidate_queries:
    baseline = baseline_retrieve(query, top_k=10)
    reranked = rerank_results(query, baseline)
    baseline_top3 = [r["chunk_id"] for r in baseline[:3]]
    reranked_top3 = [r["chunk_id"] for r in reranked[:3]]

    # Chunks selected only after re-ranking
    promoted = [
        r for r in reranked[:3]
        if r["chunk_id"] not in baseline_top3]
    experiment_results[query] = {
        "baseline": baseline,
        "reranked": reranked,
        "promoted": promoted}
    print("\n" + "=" * 80)
    print("QUERY:", query)
    print("Baseline top 3:", baseline_top3)
    print("Re-ranked top 3:", reranked_top3)
    if promoted:
        print("Newly promoted chunks:")
        for r in promoted:
            old_rank = next(
                x["rank"] for x in baseline
                if x["chunk_id"] == r["chunk_id"])
            print(
                f"  Chunk {r['chunk_id']}: "
                f"rank {old_rank} -> {r['rerank_position']}, "
                f"page {r['page']}")
    else:
        print("No new chunk entered the top 3.")



QUERY: What should an organization do when AI risks exceed its risk tolerance?
Baseline top 3: [31, 30, 43]
Re-ranked top 3: [30, 31, 33]
Newly promoted chunks:
  Chunk 33: rank 10 -> 3, page 12

QUERY: How should AI systems be monitored after deployment, and what happens when risks emerge?
Baseline top 3: [55, 25, 28]
Re-ranked top 3: [25, 114, 36]
Newly promoted chunks:
  Chunk 114: rank 4 -> 2, page 36
  Chunk 36: rank 9 -> 3, page 13

QUERY: What is the difference between measuring AI risks and managing them?
Baseline top 3: [143, 25, 22]
Re-ranked top 3: [28, 23, 25]
Newly promoted chunks:
  Chunk 28: rank 4 -> 1, page 11
  Chunk 23: rank 6 -> 2, page 10

QUERY: How should organizations handle risks that cannot be adequately measured?
Baseline top 3: [32, 31, 25]
Re-ranked top 3: [23, 25, 24]
Newly promoted chunks:
  Chunk 23: rank 10 -> 1, page 10
  Chunk 24: rank 7 -> 3, page 10

QUERY: What responsibilities do senior leaders have for AI risk governance?
Baseline top 3: [81, 86

In [42]:

queries_to_inspect = [
    "How should organizations handle risks that cannot be adequately measured?",
    "How should AI systems be monitored after deployment, and what happens when risks emerge?",
    "How should an organization address risks to people who are not direct users of an AI system?",
    "What should happen when an AI system fails to achieve its intended objectives?"
]

for query in queries_to_inspect:
    result = experiment_results[query]
    baseline = result["baseline"]
    reranked = result["reranked"]

    print("\n" + "=" * 100)
    print("QUERY:", query)

    print("\n--- BASELINE TOP 3 ---")
    for item in baseline[:3]:
        print(
            f"\nRank {item['rank']} | Chunk {item['chunk_id']} "
            f"| Page {item['page']} | Cosine: {item['score']:.4f}"
        )
        print(item["text"])

    print("\n--- RE-RANKED TOP 3 ---")
    for item in reranked[:3]:
        original_rank = next(
            x["rank"] for x in baseline
            if x["chunk_id"] == item["chunk_id"]
        )
        print(
            f"\nOriginal rank {original_rank} -> "
            f"New rank {item['rerank_position']} | "
            f"Chunk {item['chunk_id']} | Page {item['page']}"
        )
        print(item["text"])



QUERY: How should organizations handle risks that cannot be adequately measured?

--- BASELINE TOP 3 ---

Rank 1 | Chunk 32 | Page 12 | Cosine: 0.6646
tolerance, and
response established by organizational, domain, discipline, sector, or professional
requirements. Some sectors or industries may have established definitions of harm or
established documentation, reporting, and disclosure requirements. Within sectors,
risk management may depend on existing guidelines for specific applications and
use case settings. Where established guidelines do not exist, organizations should
define reasonable risk tolerance. Once tolerance is defined, this AI RMF can be used
to manage risks and to document risk management processes.
1.2.3
Risk Prioritization
Attempting to eliminate negative risk entirely can be counterproductive in practice because
not all incidents and failures can be eliminated. Unrealistic expectations about risk may
lead organizations to allocate resources in a manner that makes ri

In [43]:

second_case_queries = [
    "What should an organization do when residual AI risks remain after risk treatment?",
    "What should organizations do if deployed AI systems develop new risks over time?",
    "How should organizations respond when monitoring detects that an AI system is no longer performing as intended?",
    "What does the framework recommend when an AI system cannot detect or correct its own errors?"
]

for query in second_case_queries:
    baseline = baseline_retrieve(query, top_k=10)
    reranked = rerank_results(query, baseline)

    baseline_ids = [r["chunk_id"] for r in baseline[:3]]

    print("\n" + "=" * 85)
    print("QUERY:", query)
    print("Baseline top 3:", baseline_ids)
    print("Re-ranked top 3:", [r["chunk_id"] for r in reranked[:3]])

    print("\nRe-ranked results:")
    for r in reranked[:3]:
        old_rank = next(
            x["rank"] for x in baseline
            if x["chunk_id"] == r["chunk_id"]
        )
        print(
            f"Chunk {r['chunk_id']}: #{old_rank} -> "
            f"#{r['rerank_position']} | Page {r['page']}"
        )
        print(r["text"][:500])



QUERY: What should an organization do when residual AI risks remain after risk treatment?
Baseline top 3: [36, 25, 35]
Re-ranked top 3: [36, 35, 31]

Re-ranked results:
Chunk 36: #1 -> #1 | Page 13
tions.
Residual risk – defined as risk remaining after risk treatment (Source: ISO GUIDE 73) –
directly impacts end users or affected individuals and communities. Documenting residual
risks will call for the system provider to fully consider the risks of deploying the AI product
and will inform end users about potential negative impacts of interacting with the system.
1.2.4
Organizational Integration and Management of Risk
AI risks should not be considered in isolation. Different AI actors have 
Chunk 35: #3 -> #2 | Page 13
re designed or deployed to directly
interact with humans as compared to AI systems that are not. Higher initial prioritization
may be called for in settings where the AI system is trained on large datasets comprised of
sensitive or protected data such as personally ident

In [44]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import time

GEN_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)

gen_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

gen_model.eval()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [49]:

def generate_answer(query, retrieved_chunks):
    context = "\n\n".join(
        f"[Page {item['page']} | Chunk {item['chunk_id']}]\n"
        f"{item['text'][:500]}"
        for item in retrieved_chunks[:3])

    prompt = f"""Question: {query}
Answer the question using only the context provided.
Give a direct answer in 2 to 4 sentences.
Include relevant actions, conditions, or exceptions.
Do not invent information. If the context is insufficient, say so.

Context:
{context}

Answer:
"""

    inputs = gen_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=768
    ).to(next(gen_model.parameters()).device)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.no_grad():
        output = gen_model.generate(
            **inputs,
            max_new_tokens=160,
            do_sample=False,
            num_beams=4,
            early_stopping=True,
            min_new_tokens=20
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start
    new_tokens = output[0][inputs["input_ids"].shape[-1]:]
    answer = gen_tokenizer.decode(
        new_tokens,
        skip_special_tokens=True).strip()
    return answer, elapsed


selected_queries = [
    "How should organizations handle risks that cannot be adequately measured?",
    "How should organizations respond when monitoring detects that an AI system is no longer performing as intended?"]

answer_results = {}
for query in selected_queries:
    baseline = baseline_retrieve(query, top_k=10)
    reranked = rerank_results(query, baseline)
    answer1, time1 = generate_answer(query, baseline[:3])
    answer2, time2 = generate_answer(query, reranked[:3])
    answer_results[query] = {
        "baseline_answer": answer1,
        "reranked_answer": answer2,
        "baseline_generation_time": time1,
        "reranked_generation_time": time2,
        "baseline_chunks": [r["chunk_id"] for r in baseline[:3]],
        "reranked_chunks": [r["chunk_id"] for r in reranked[:3]]
    }

    print("\n" + "=" * 80)
    print("QUERY:", query)

    print("\nBASELINE TOP 3:", answer_results[query]["baseline_chunks"])
    print("RE-RANKED TOP 3:", answer_results[query]["reranked_chunks"])

    print("\nANSWER 1 — BASELINE")
    print(answer1)

    print("\nANSWER 2 — RE-RANKED")
    print(answer2)

    print(f"\nBaseline generation: {time1:.3f} seconds")
    print(f"Re-ranked generation: {time2:.3f} seconds")



QUERY: How should organizations handle risks that cannot be adequately measured?

BASELINE TOP 3: [32, 31, 25]
RE-RANKED TOP 3: [23, 25, 24]

ANSWER 1 — BASELINE
To address risks that cannot be adequately measured, organizations should define a reasonable risk tolerance based on their specific priorities and resource considerations. Where established guidelines do not exist, organizations should establish their own risk management framework. This framework should include internal governance structures and technical safeguards to manage risk in AI systems. Tracking emergent risks is crucial for enhancing risk management efforts, and techniques for measuring these risks should be considered. Regardless of whether challenges for specifying AI risk tolerances remain unresolved, all parties and AI actors should manage risk in their AI systems.

ANSWER 2 — RE-RANKED
To address risks that cannot be adequately measured, organizations should focus on enhancing their risk management efforts. Th

In [54]:
import time
import torch

def synchronize_gpu():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

query = "What procedures should be followed when a previously unknown AI risk is identified?"

# 1. Bi-encoder retrieval: retrieve top 10
synchronize_gpu()
start = time.perf_counter()

candidates = baseline_retrieve(query, top_k=10)

synchronize_gpu()
retrieval_time = time.perf_counter() - start

# 2. Cross-encoder reranking: rerank the same 10 candidates
synchronize_gpu()
start = time.perf_counter()

reranked = rerank_results(query, candidates)

synchronize_gpu()
reranking_time = time.perf_counter() - start

# 3. Report latency
print(f"Bi-encoder retrieval: {retrieval_time:.4f} seconds")
print(f"Cross-encoder reranking: {reranking_time:.4f} seconds")
print(f"Combined retrieval + reranking: {retrieval_time + reranking_time:.4f} seconds")

Bi-encoder retrieval: 0.0111 seconds
Cross-encoder reranking: 0.0356 seconds
Combined retrieval + reranking: 0.0466 seconds


In [50]:

query = "What procedures should be followed when a previously unknown AI risk is identified?"

baseline = baseline_retrieve(query, top_k=10)
reranked = rerank_results(query, baseline)

answer_baseline, time_baseline = generate_answer(query, baseline[:3])
answer_reranked, time_reranked = generate_answer(query, reranked[:3])

print("QUERY:", query)
print("BASELINE TOP 3:", [x["chunk_id"] for x in baseline[:3]])
print("RERANKED TOP 3:", [x["chunk_id"] for x in reranked[:3]])

print("\nBASELINE ANSWER:\n", answer_baseline)
print("\nRERANKED ANSWER:\n", answer_reranked)

print(f"\nBaseline generation: {time_baseline:.3f} seconds")
print(f"Reranked generation: {time_reranked:.3f} seconds")


QUERY: What procedures should be followed when a previously unknown AI risk is identified?
BASELINE TOP 3: [36, 23, 25]
RERANKED TOP 3: [111, 55, 25]

BASELINE ANSWER:
 When a previously unknown AI risk is identified, the following procedures should be followed:

1. Document the risk: Fully consider the risks of deploying the AI product and inform end users about potential negative impacts of interacting with the system.

2. Integrate risk management: AI risks should not be considered in isolation; different AI actors need to work together to manage these risks effectively.

3. Manage risks: All parties and AI actors should manage risk in the AI systems they develop, deploy, or use as standalone or integrated components.

4. Track emergent risks: Organizations' risk management efforts will be enhanced by identifying and tracking emergent risks and considering techniques for measuring them.

RERANKED ANSWER:
 When a previously unknown AI risk is identified, the following procedures shou